# Contextual and Non‑Contextual Decomposition of Molecular Hamiltonian

This notebook demonstrates how to generate a molecular Hamiltonian using **OpenFermion**, and how to separate that Hamiltonian into contextual and non‑contextual parts using the *contextual subspace* techniques from the `cs_vqe` module (as described in the `CS_VQE_how_to_use.ipynb` notebook).

Given a molecular geometry, we

1. Generate the fermionic and qubit Hamiltonians via `openfermionpyscf` and the Jordan‑Wigner transform.
2. Convert the qubit Hamiltonian into a simple dictionary format expected by the `cs_vqe` routines.
3. Use `cs_vqe.greedy_dfs` to approximate a large non‑contextual sub‑Hamiltonian and then extract the corresponding contextual part.
4. Convert the contextual part back into the standard OpenFermion formats (`FermionOperator` and `QubitOperator`).

This workflow follows the approach outlined in the `ContextualSubspaceVQE` repository – particularly the functions `contextualQ_ham` and `greedy_dfs` described by Kirby \_et al.\_


## Prerequisites
Before running this notebook you need to install the following packages in your Python environment:

- `openfermion`: The quantum simulation library for chemistry.
- `openfermionpyscf`: Provides a wrapper for PySCF to compute molecular integrals.
- `cs_vqe`: The Contextual Subspace VQE package (see the repository [wmkirby1/ContextualSubspaceVQE](https://github.com/wmkirby1/ContextualSubspaceVQE)).

If any of these packages are not installed, install them using `pip install openfermion openfermionpyscf contextual-subspace-vqe` or by cloning the respective GitHub repositories and adding them to your `PYTHONPATH` as shown in the official tutorial.

In [ ]:
# Define the molecular geometry.
# Example: H2 molecule with two hydrogen atoms separated by 0.74 Å.
# You can modify the geometry, basis, multiplicity and charge as needed.

from openfermion import MolecularData
from openfermionpyscf import run_pyscf
from openfermion.transforms import jordan_wigner

# geometry is a list of tuples: (atom symbol, (x, y, z) in atomic units)
geometry = [
    ('C', (-0.288346, 0.723178, 0.872948)),
    ('C', (-0.288249, -0.722959, 0.873144)),
    ('H', (-1.280153, 1.002851, 0.142283)),
    ('H', (-0.492354, 1.301868, 1.770883)),
    ('H', (-1.279406, -1.003258, 0.142460)),
    ('H', (-0.492048, -1.301456, 1.771243)),
    ('O', (0.874871, 1.114520, 0.203874)),
    ('O', (0.875179, -1.114451, 0.204365)),
    ('C', (1.552997, -0.000009, -0.222967)),
    ('O', (2.554737, 0.000033, -0.848568)),
    ('O', (-2.296902, -0.655858, -0.590465)),
    ('O', (-2.297190, 0.655597, -0.589909))
]


basis = 'sto-3g'
multiplicity = 1
charge = 0

# Create a MolecularData object and run PySCF to compute the integrals.
molecule = MolecularData(geometry, basis, multiplicity, charge)
molecule = run_pyscf(molecule, run_scf=True, run_mp2=False, run_cisd=False, run_ccsd=False, run_fci=False)

# Obtain the fermionic Hamiltonian and map it to a qubit Hamiltonian via Jordan‑Wigner.
fermion_ham = molecule.get_molecular_hamiltonian(
        occupied_indices=list(range(25)),
        active_indices=list(range(25, 35)),
    )
qubit_ham = jordan_wigner(fermion_ham)

# Display the qubit Hamiltonian.
print('Qubit Hamiltonian:')
print(qubit_ham)


Qubit Hamiltonian:
-479.7500757146952 [] +
-0.0017418922117838401 [X0 X1 Y2 Y3] +
-4.669386845188719e-06 [X0 X1 Y2 Z3 Z4 Y5] +
-5.409177615832209e-06 [X0 X1 Y2 Z3 Z4 Z5 Z6 Y7] +
7.499584982786828e-07 [X0 X1 Y2 Z3 Z4 Z5 Z6 Z7 Z8 Y9] +
-0.0005326181932715269 [X0 X1 Y2 Z3 Z4 Z5 Z6 Z7 Z8 Z9 Z10 Y11] +
1.5142616497866939e-06 [X0 X1 Y2 Z3 Z4 Z5 Z6 Z7 Z8 Z9 Z10 Z11 Z12 Y13] +
-0.00020267712037710242 [X0 X1 Y2 Z3 Z4 Z5 Z6 Z7 Z8 Z9 Z10 Z11 Z12 Z13 Z14 Y15] +
0.0016948772377135585 [X0 X1 Y2 Z3 Z4 Z5 Z6 Z7 Z8 Z9 Z10 Z11 Z12 Z13 Z14 Z15 Z16 Y17] +
0.00017009800717990287 [X0 X1 Y2 Z3 Z4 Z5 Z6 Z7 Z8 Z9 Z10 Z11 Z12 Z13 Z14 Z15 Z16 Z17 Z18 Y19] +
-4.669386845188718e-06 [X0 X1 X3 X4] +
-5.409177615832209e-06 [X0 X1 X3 Z4 Z5 X6] +
7.499584982786828e-07 [X0 X1 X3 Z4 Z5 Z6 Z7 X8] +
-0.0005326181932715269 [X0 X1 X3 Z4 Z5 Z6 Z7 Z8 Z9 X10] +
1.5142616497866939e-06 [X0 X1 X3 Z4 Z5 Z6 Z7 Z8 Z9 Z10 Z11 X12] +
-0.0002026771203771024 [X0 X1 X3 Z4 Z5 Z6 Z7 Z8 Z9 Z10 Z11 Z12 Z13 X14] +
0.0016948772377135585 [X0 X1 

In [7]:
# Helper: convert an OpenFermion QubitOperator to a dictionary of Pauli strings.
def qubit_operator_to_dict(qubit_op):
    """Return a dict mapping Pauli strings to coefficients.
    The Pauli string is a string over 
        {'I', 'X', 'Y', 'Z'} of length equal to the number of qubits.
    Example: QubitOperator("X0 Y2", 0.5) on 3 qubits becomes {'XIY': 0.5}.
    """
    # Determine number of qubits by inspecting the largest qubit index in the operator.
    n_qubits = 0
    for term in qubit_op.terms:
        for qubit_index, pauli in term:
            if qubit_index + 1 > n_qubits:
                n_qubits = qubit_index + 1

    ham_dict = {}
    for term, coeff in qubit_op.terms.items():
        pauli_list = ['I'] * n_qubits
        for qubit_index, pauli in term:
            pauli_list[qubit_index] = pauli
        pauli_str = ''.join(pauli_list)
        # Combine coefficients if the same Pauli string appears multiple times.
        ham_dict[pauli_str] = ham_dict.get(pauli_str, 0.0) + coeff
    return ham_dict

# Convert the qubit Hamiltonian to the dictionary format used by cs_vqe.
ham_dict = qubit_operator_to_dict(qubit_ham)

# Inspect the dictionary representation.
print('Hamiltonian as dictionary (Pauli string -> coefficient):')
for pauli_str, coeff in ham_dict.items():
    print(f'{coeff:+.6f} 	 {pauli_str}')


Hamiltonian as dictionary (Pauli string -> coefficient):
-479.750076 	 IIIIIIIIIIIIIIIIIIII
+0.454506 	 ZIIIIIIIIIIIIIIIIIII
+0.454506 	 IZIIIIIIIIIIIIIIIIII
+0.430120 	 IIZIIIIIIIIIIIIIIIII
+0.430120 	 IIIZIIIIIIIIIIIIIIII
+0.390490 	 IIIIZIIIIIIIIIIIIIII
+0.390490 	 IIIIIZIIIIIIIIIIIIII
+0.433471 	 IIIIIIZIIIIIIIIIIIII
+0.433471 	 IIIIIIIZIIIIIIIIIIII
+0.365262 	 IIIIIIIIZIIIIIIIIIII
+0.365262 	 IIIIIIIIIZIIIIIIIIII
+0.371868 	 IIIIIIIIIIZIIIIIIIII
+0.371868 	 IIIIIIIIIIIZIIIIIIII
+0.270303 	 IIIIIIIIIIIIZIIIIIII
+0.270303 	 IIIIIIIIIIIIIZIIIIII
+0.238969 	 IIIIIIIIIIIIIIZIIIII
+0.238969 	 IIIIIIIIIIIIIIIZIIII
+0.077354 	 IIIIIIIIIIIIIIIIZIII
+0.077354 	 IIIIIIIIIIIIIIIIIZII
-0.048022 	 IIIIIIIIIIIIIIIIIIZI
-0.048022 	 IIIIIIIIIIIIIIIIIIIZ
+0.114068 	 ZZIIIIIIIIIIIIIIIIII
-0.000002 	 XZXIIIIIIIIIIIIIIIII
-0.000002 	 YZYIIIIIIIIIIIIIIIII
+0.065974 	 ZIZIIIIIIIIIIIIIIIII
+0.067716 	 ZIIZIIIIIIIIIIIIIIII
+0.004184 	 XZZZXIIIIIIIIIIIIIII
+0.004184 	 YZZZYIIIIIIIIIIIIIII
+0.076387 	 ZIIIZ

In [9]:
# Apply contextual subspace routines to split the Hamiltonian.
# We approximate a large non‑contextual sub‑Hamiltonian using greedy depth‑first search.

import cs_vqe as c

# The greedy_dfs function takes the Hamiltonian dictionary, a timeout (in seconds),
# and an optional criterion ('weight' or 'size').
# It returns a list of candidate noncontextual subsets of Pauli terms; the last element
# in this list is the best candidate found within the time limit.
noncon_candidates = c.greedy_dfs(ham_dict, cutoff=1, criterion='weight')

# Choose the final candidate (largest total weight) as the noncontextual subset.
noncon_terms = set(noncon_candidates[-1])
print(f"Number of terms in noncontextual subset: {len(noncon_terms)}")

# Construct the noncontextual and contextual Hamiltonians in dictionary form.
noncon_dict = {pauli: ham_dict[pauli] for pauli in noncon_terms}
contextual_dict = {pauli: ham_dict[pauli] for pauli in ham_dict.keys() if pauli not in noncon_terms}

print('Non‑contextual Hamiltonian:')
for pauli, coeff in noncon_dict.items():
    print(f'{coeff:+.6f} 	 {pauli}')

print('Contextual Hamiltonian:')
for pauli, coeff in contextual_dict.items():
    print(f'{coeff:+.6f} 	 {pauli}')


Number of terms in noncontextual subset: 249
Non‑contextual Hamiltonian:
+0.061249 	 IIIIIIIIIIZIIIZIIIII
+0.049308 	 IIIIIIIIIIIIIIIZIIIZ
+0.071173 	 IIIIIIIIIIIIIIIZIZII
+0.057056 	 IIIZIIIIIIIIIIIIIIIZ
+0.074860 	 IIIIIIIZIIIZIIIIIIII
+0.049308 	 IIIIIIIIIIIIIIZIIIZI
+0.077783 	 ZIIIIIIIIIIZIIIIIIII
+0.074514 	 IIIIIIIIIIIZIIIIIZII
+0.072920 	 IIIIIZIIIIZIIIIIIIII
+0.052591 	 IIIIIIIIZIIIIIIIIIZI
+0.080299 	 ZIIIIZIIIIIIIIIIIIII
+0.007209 	 IIIXZIZZZZZXIIIIIIII
+0.048760 	 IIIIIIZIIIIIIIIIIIIZ
+0.365262 	 IIIIIIIIZIIIIIIIIIII
+0.069290 	 IIIIIZIIIIIIIIIIIIZI
+0.032603 	 IIIIIIIIIZIIIIIZIIII
+0.110550 	 IIIIIIIIZIIIZIIIIIII
+0.061849 	 IIIIIIIIIIIIIZIIZIII
+0.080102 	 IIIIIZIIIIIIIIIIZIII
+0.060062 	 IIIIIIIIIIIIZIIIZIII
-0.016786 	 IIIYZZZZZZZYIIIZIIII
+0.008677 	 IIIYZZZZZZZYIIIIIIIZ
-0.031447 	 IIIXZZZIZZZXIIIIIIII
+0.071129 	 IIIIIIZIIIIIIIIIIZII
+0.077835 	 IIIIIIIIIIIZIIIIZIII
+0.077142 	 IIIIZIIIIIIIIIIIZIII
+0.061849 	 IIIIIIIIIIIIZIIIIZII
-0.014379 	 IZIXZZZZZZZXIIIIIIII
+0.

In [10]:
# Convert the contextual part back into QubitOperator and FermionOperator forms.
from openfermion import QubitOperator
from openfermion.transforms.opconversions import reverse_jordan_wigner

# Build a QubitOperator from the contextual dictionary.
contextual_qubit_op = QubitOperator()
for pauli_str, coeff in contextual_dict.items():
    # Create a list of (qubit_index, pauli) tuples for non‑identity entries.
    term_list = []
    for index, pauli in enumerate(pauli_str):
        if pauli != 'I':
            term_list.append((index, pauli))
    # Use the QubitOperator constructor to add this term.
    contextual_qubit_op += QubitOperator(tuple(term_list), coeff)

# Use the reverse Jordan‑Wigner transform to obtain a FermionOperator.
contextual_fermion_op = reverse_jordan_wigner(contextual_qubit_op)

print('Reduced contextual Hamiltonian (QubitOperator):')
print(contextual_qubit_op)

print('Reduced contextual Hamiltonian (FermionOperator):')
print(contextual_fermion_op)


Reduced contextual Hamiltonian (QubitOperator):
-0.0017418922117838401 [X0 X1 Y2 Y3] +
-4.669386845188719e-06 [X0 X1 Y2 Z3 Z4 Y5] +
-5.409177615832209e-06 [X0 X1 Y2 Z3 Z4 Z5 Z6 Y7] +
7.499584982786828e-07 [X0 X1 Y2 Z3 Z4 Z5 Z6 Z7 Z8 Y9] +
-0.0005326181932715269 [X0 X1 Y2 Z3 Z4 Z5 Z6 Z7 Z8 Z9 Z10 Y11] +
1.5142616497866939e-06 [X0 X1 Y2 Z3 Z4 Z5 Z6 Z7 Z8 Z9 Z10 Z11 Z12 Y13] +
-0.00020267712037710242 [X0 X1 Y2 Z3 Z4 Z5 Z6 Z7 Z8 Z9 Z10 Z11 Z12 Z13 Z14 Y15] +
0.0016948772377135585 [X0 X1 Y2 Z3 Z4 Z5 Z6 Z7 Z8 Z9 Z10 Z11 Z12 Z13 Z14 Z15 Z16 Y17] +
0.00017009800717990287 [X0 X1 Y2 Z3 Z4 Z5 Z6 Z7 Z8 Z9 Z10 Z11 Z12 Z13 Z14 Z15 Z16 Z17 Z18 Y19] +
-4.669386845188718e-06 [X0 X1 X3 X4] +
-5.409177615832209e-06 [X0 X1 X3 Z4 Z5 X6] +
7.499584982786828e-07 [X0 X1 X3 Z4 Z5 Z6 Z7 X8] +
-0.0005326181932715269 [X0 X1 X3 Z4 Z5 Z6 Z7 Z8 Z9 X10] +
1.5142616497866939e-06 [X0 X1 X3 Z4 Z5 Z6 Z7 Z8 Z9 Z10 Z11 X12] +
-0.0002026771203771024 [X0 X1 X3 Z4 Z5 Z6 Z7 Z8 Z9 Z10 Z11 Z12 Z13 X14] +
0.0016948772377135585 [X

In [13]:
# Build a QubitOperator from the contextual dictionary.
noncontextual_qubit_op = QubitOperator()
for pauli_str, coeff in noncon_dict.items():
    # Create a list of (qubit_index, pauli) tuples for non‑identity entries.
    term_list = []
    for index, pauli in enumerate(pauli_str):
        if pauli != 'I':
            term_list.append((index, pauli))
    # Use the QubitOperator constructor to add this term.
    noncontextual_qubit_op += QubitOperator(tuple(term_list), coeff)

# Use the reverse Jordan‑Wigner transform to obtain a FermionOperator.
noncontextual_fermion_op = reverse_jordan_wigner(noncontextual_qubit_op)

print('Reduced contextual Hamiltonian (QubitOperator):')
print(noncontextual_qubit_op)

print('Reduced contextual Hamiltonian (FermionOperator):')
print(noncontextual_fermion_op)

Reduced contextual Hamiltonian (QubitOperator):
-479.7500757146952 [] +
0.4545063890369956 [Z0] +
0.11406753792150878 [Z0 Z1] +
0.06597361521803174 [Z0 Z2] +
-0.014911295433917572 [Z0 X3 Z4 Z5 Z6 Z7 Z8 Z9 Z10 X11] +
-0.014911295433917572 [Z0 Y3 Z4 Z5 Z6 Z7 Z8 Z9 Z10 Y11] +
0.06771550742981558 [Z0 Z3] +
0.07638688625600822 [Z0 Z4] +
0.0802992714534253 [Z0 Z5] +
0.06771188034963405 [Z0 Z6] +
0.08614512540116702 [Z0 Z7] +
0.037217255381980424 [Z0 Z8] +
0.0373304632370674 [Z0 Z9] +
0.07432897975435053 [Z0 Z10] +
0.07778302790993243 [Z0 Z11] +
0.048669956634446454 [Z0 Z12] +
0.04942214833185349 [Z0 Z13] +
0.081725361456439 [Z0 Z14] +
0.08329483582722791 [Z0 Z15] +
0.08398941081726127 [Z0 Z16] +
0.09325634583274531 [Z0 Z17] +
0.06259537118703605 [Z0 Z18] +
0.0638154934878723 [Z0 Z19] +
0.45450638903699603 [Z1] +
0.06771550742981558 [Z1 Z2] +
-0.014378677240646044 [Z1 X3 Z4 Z5 Z6 Z7 Z8 Z9 Z10 X11] +
-0.014378677240646044 [Z1 Y3 Z4 Z5 Z6 Z7 Z8 Z9 Z10 Y11] +
0.06597361521803174 [Z1 Z3] +
0.0802

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from openfermion import get_sparse_operator, count_qubits
from scipy.sparse.linalg import eigsh


def get_lowest_eigenvalues_qubit(qubit_op, k=10, n_qubits=None, tol=1e-10):
    """
    Compute the lowest k eigenvalues of a QubitOperator efficiently.

    Parameters
    ----------
    qubit_op : openfermion.QubitOperator
        Hamiltonian in qubit form.
    k : int
        Number of lowest eigenvalues to compute.
    n_qubits : int or None
        Number of qubits. If None, inferred from qubit_op.
    tol : float
        Tolerance for sparse eigensolver.

    Returns
    -------
    evals : np.ndarray
        Sorted lowest eigenvalues.
    """
    if n_qubits is None:
        n_qubits = count_qubits(qubit_op)

    H_sparse = get_sparse_operator(qubit_op, n_qubits=n_qubits).tocsr()
    dim = H_sparse.shape[0]

    # If matrix is tiny, just diagonalize exactly.
    if dim <= 64 or k >= dim - 1:
        evals = np.linalg.eigvalsh(H_sparse.toarray())
        return np.sort(evals)[:k]

    # Sparse lowest-eigenvalue solve
    evals = eigsh(H_sparse, k=k, which="SA", return_eigenvectors=False, tol=tol)
    return np.sort(evals)


def plot_spectrum(evals, title="Low-energy spectrum", ylabel="Energy"):
    """
    Simple spectrum plot.
    """
    plt.figure(figsize=(7, 4))
    plt.scatter(range(len(evals)), evals)
    plt.xlabel("Level index")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

low_evals = get_lowest_eigenvalues_qubit(noncontextual_qubit_op, k=12)
print("Lowest eigenvalues:")
print(low_evals)

plot_spectrum(low_evals, title="Lowest 12 eigenvalues of noncontextual Hamiltonian")

## Data Flow Summary

- **Fermionic Hamiltonian**: Created from `MolecularData` using PySCF and stored as an `openfermion.FermionOperator`.
- **Qubit Hamiltonian**: Obtained by applying the Jordan‑Wigner transform (`jordan_wigner`) to the fermionic Hamiltonian; represented as an `openfermion.QubitOperator`.
- **Pauli dictionary**: For compatibility with the `cs_vqe` routines, the qubit Hamiltonian is mapped to a Python `dict` where each key is a Pauli string (e.g., `'XIZY'`) and each value is the corresponding coefficient.
- **Non‑contextual subset**: Using `cs_vqe.greedy_dfs`, a subset of Pauli strings that forms a large non‑contextual sub‑Hamiltonian is found.  This subset is represented as a set of keys from the Pauli dictionary.
- **Contextual part**: The contextual Hamiltonian consists of the remaining Pauli terms in the dictionary.
- **Reduced Hamiltonians**: The contextual dictionary is converted back into an OpenFermion `QubitOperator`.  Applying `openfermion.transforms.opconversions.reverse_jordan_wigner` maps this qubit operator back into a `FermionOperator` so that the contextual (quantum) Hamiltonian can be passed into later quantum algorithms.

The non‑contextual part can be handled with classical methods (quasi‑quantized models), while the contextual part defines the reduced Hamiltonian for the quantum computer.